In [ ]:
import json
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

from data.dataset import StarryNPZDataset
from trainers.lc2img_module import LC2ImgModule


In [ ]:
OUTPUT_DIR="./output_Apr28"

DATA_PATH="./output_Apr28/validation_set/val_set.npz"

In [ ]:
# Load model

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 1) Load NN model
# ────────────────────────────────────────────────────────────────────────────
# OUTPUT_DIR = "./output_Apr28"  # or wherever you trained
with open(os.path.join(OUTPUT_DIR, "config.json"), "r") as f:
    cfg = json.load(f)

MODEL_PATH     = os.path.join(OUTPUT_DIR, "final_model.pt")
# DATA_PATH      = cfg["data_dir"]
IMG_SIZE       = cfg["img_size"]
LATENT_CHANS   = cfg["latent_channels"]
GRID_SIZE      = (cfg["grid_h"], cfg["grid_w"])
BASE_CHANNELS  = cfg["base_channels"]
NUM_PYRAMID    = cfg.get("num_pyramid", 3)
USE_RESIDUALS  = cfg.get("use_residuals", False)
RES_DILATIONS  = cfg.get("res_dilations", [1,2,4,8])
MASK_CORNERS   = cfg.get("mask_corners", False)
PERCEPTUAL_LOSS = cfg.get("use_perceptual_loss", True)
USE_SSIM        = cfg.get("use_ssim_loss", True)
LAMBDA_PERC     = cfg.get("lambda_perc", 0)
LAMBDA_SSIM     = cfg.get("lambda_ssim", 0)

print("Loaded config:", cfg)
print("Model path:", MODEL_PATH)
print("Data path:", DATA_PATH)
print("Image size:", IMG_SIZE)
print("Grid size:", GRID_SIZE)
print("Latent channels:", LATENT_CHANS)
print("Base channels:", BASE_CHANNELS)
print("Num pyramid:", NUM_PYRAMID)
print("Use residuals:", USE_RESIDUALS)
print("Residual dilations:", RES_DILATIONS)
print("Mask corners:", MASK_CORNERS)
print("Use perceptual loss:", PERCEPTUAL_LOSS)
print("Use SSIM loss:", USE_SSIM)
print("Lambda perceptual:", LAMBDA_PERC)
print("Lambda SSIM:", LAMBDA_SSIM)

# ────────────────────────────────────────────────────────────────────────────
# 2) Instantiate & load the model using the same args
# ────────────────────────────────────────────────────────────────────────────

model = LC2ImgModule(
    lr               = cfg["lr"],
    latent_channels  = LATENT_CHANS,
    grid_size        = GRID_SIZE,
    img_size         = IMG_SIZE,
    base_channels    = BASE_CHANNELS,
    num_pyramid      = NUM_PYRAMID,
    use_residuals    = USE_RESIDUALS,
    res_dilations    = RES_DILATIONS,
    mask_corners     = MASK_CORNERS,

    # ← these two must match your training run
    use_perceptual_loss = PERCEPTUAL_LOSS,
    use_ssim_loss       = USE_SSIM,
    lambda_perc         = LAMBDA_PERC,
    lambda_ssim         = LAMBDA_SSIM,
)


model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

# ────────────────────────────────────────────────────────────────────────────
# 3) Load dataset & run inference just like before
# ────────────────────────────────────────────────────────────────────────────
dataset = StarryNPZDataset(
    path     = DATA_PATH,
    lc_key   = 'flux',
    img_key  = 'image',
    img_size = IMG_SIZE
)

n=8
indices = np.random.choice(len(dataset), size=n, replace=False)
fig, axes = plt.subplots(n, 2, figsize=(8, 16))

with torch.no_grad():
    for row, idx in enumerate(indices):
        lc, true_img = dataset[idx]
        pred_img = model(lc.unsqueeze(0)).squeeze().cpu().numpy()

        axes[row, 0].imshow(true_img.squeeze(), cmap='plasma')
        axes[row, 0].set_title(f"True #{idx}")
        axes[row, 0].axis('off')

        axes[row, 1].imshow(pred_img.squeeze(), cmap='plasma')
        axes[row, 1].set_title(f"Pred #{idx}")
        axes[row, 1].axis('off')

plt.tight_layout()
plt.show()
